# Варіант 1: Кліматичні дані (м. Київ, 2021–2024 рр.)
Практичне заняття: Агрегація даних (groupby), зведені таблиці (pivot_table, pivot), крос-табуляція (crosstab) та Tidy Data

### Підготовка даних: Створення довгого (Tidy) набору `climate` (48 рядків)

In [ ]:
import numpy as np
import pandas as pd

# Фіксація seed за номером варіанта
np.random.seed(1)

years = [2021, 2022, 2023, 2024]
months = list(range(1, 13))

# Базовий місячний кліматичний профіль (°C)
base_temp = [-3.0, -2.0, 3.0, 10.0, 16.0, 19.5, 21.5, 20.5, 15.0, 9.0, 3.0, -1.5]

data = []
for y_idx, y in enumerate(years):
    for m_idx, m in enumerate(months):
        # Додаємо легкий тренд потепління (+0.2°C за рік) + випадкові коливання
        # Вища нестабільність (шум) для перехідних місяців (березень, квітень, листопад)
        noise_std = 2.5 if m in [3, 4, 11] else 1.0
        temp = round(base_temp[m_idx] + y_idx * 0.25 + np.random.normal(0, noise_std), 1)
        data.append({"рік": y, "місяць": m, "температура": temp})

climate = pd.DataFrame(data)

print(f"Розмірність датасету: {climate.shape} (очікується 48 рядків, 3 стовпці)")
display(climate.head(12))

### Завдання 1. groupby і agg за роками

In [ ]:
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
display(yearly_stats)

**Аналіз річного тренду:**
Середньорічна температура поступово зростає ($2021\text{ рік: } 8.91^\circ\text{C} \rightarrow 2024\text{ рік: } 9.68^\circ\text{C}$), як і максимальні значення. У цих даних прослідковується **помітний тренд до потепління**, хоча міжрічні коливання мінімальних температур показують локальну варіативність.

### Завдання 2. groupby і agg за місяцями

In [ ]:
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
display(monthly_stats)

unstable_month = monthly_stats["std"].idxmax()
max_std_val = monthly_stats["std"].max()

print(f"\nНайбільш нестабільний місяць: {unstable_month}-й місяць (std = {max_std_val:.2f}°C)")

**Обґрунтування нестабільності:**
Найбільше стандартне відхилення ($std$) зафіксовано у **перехідні весняні/осінні місяці (наприклад, Березень/Квітень)**. 
*Припущення:* Перехідні сезони є статистично найменш стійкими, оскільки в цей період відбуваються активні перебудови атмосферної циркуляції, зміна снігового покриву та чергування арктичних вторгнень із першим інтенсивним сонячним прогрівом. Глибока зима (січень) або стабільне літо (липень) мають більш сталі баричні системи.

### Завдання 3. pivot_table

In [ ]:
pivot_climate = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
display(pivot_climate)

**Порівняння довгої (Tidy) та зведеної (Wide) форм даних:**
1. **Зведена форма (`pivot_table`):** Набагато зручніша **для читання та сприйняття людиною**, оскільки компактно відображає матрицю «12 місяців $\times$ 4 роки» без необхідності прокручувати 48 рядків. Вона дозволяє миттєво порівняти один і той самий місяць між різними роками.
2. **Довга форма (`climate`):** Ідеальна **для обчислювальної обробки, програмного групування (`groupby`), фільтрації та побудови графіків** (наприклад, у `seaborn` чи `matplotlib`). У довгій формі кожен стовпець є окремою змінною, а кожен рядок — унікальним спостереженням (принцип Tidy Data з Лекції 4).

### Завдання 4. Похідні категорії і crosstab

In [ ]:
# 1. Функція для визначення сезону
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

climate["сезон"] = climate["місяць"].apply(get_season)

# 2. Визначення загального середнього та створення булевого стовпця
overall_mean = climate["температура"].mean()
climate["тепліше_за_середнє"] = climate["температура"] > overall_mean

print(f"Загальне середнє значення температури за всі 4 роки: {overall_mean:.2f}°C\n")

# 3. Побудова crosstab
cross_tab = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
display(cross_tab)

**Перевірка розподілу частот:**
- **Літо (12 спостережень):** Усі 100% літніх місяців потрапили в категорію `True` (`тепліше_за_середнє`), що повністю відповідає фізичному змісту сезонності.
- **Зима (12 спостережень):** Усі 100% зимових місяців потрапили в категорію `False`.
- **Весна та Осінь:** Демонструють розщеплення (частина місяців, як-от травень/вересень, перевищує середньорічну планку, а березень/листопад є холоднішими за неї). Результат цілком узгоджується з очікуваннями.

### Завдання 5. pivot() на своєму наборі та його механіка

In [ ]:
# Спроба виклику .pivot() без функцій агрегації
try:
    pivoted_direct = climate.pivot(index="місяць", columns="рік", values="температура")
    print("--- Метод .pivot() успішно виконано! ---")
    display(pivoted_direct)
except Exception as e:
    print(f"Помилка виклику .pivot(): {e}")

**Пояснення успішності `.pivot()` та умов його поломки:**

1. **Чому `.pivot()` спрацював у нашому випадку?**
   На відміну від прикладу з кав'ярнями, у нашому наборі даних `climate` на кожну унікальну пару значень `(місяць, рік)` припадає **строго один рядок** (12 місяців $\times$ 4 роки = 48 унікальних комбінацій). Оскільки немає жодного дублювання координат індексу/стовпця, метод `.pivot()` однозначно знає, яке значення помістити у відповідну комірку.

2. **Яку зміну слід внести, щоб `.pivot()` завершився помилкою `ValueError`?**
   Якщо додати до даних додатковий вимір — наприклад, дані від двох різних метеостанцій одного міста (`"станція": ["Центральна", "Аеропорт"]`), то для комбінації `(місяць=1, рік=2021)` з'явиться 2 рядки з різними температурами. У такому разі `.pivot()` видасть помилку `ValueError: Index contains duplicate entries`, оскільки він не має вбудованої функції агрегації (`aggfunc`), щоб звести два числа в одне. Для такого розширеного набору доведеться використовувати `.pivot_table()`.

### Відповіді на контрольні питання:

**1. Чи можете пояснити словами логіку split-apply-combine, яку виконує groupby() — що саме відбувається на кожному з трьох кроків?**
- **Split (Розбиття):** Вхідний DataFrame ділиться на окремі підмножини (групи) на основі унікальних значень одного або кількох ключів групування (наприклад, за роком або сезоном).
- **Apply (Застосування):** До кожної утворення підгрупи незалежно застосовується обчислювальна функція (наприклад, агрегуюча функція `mean()`, `std()`, `sum()` або трансформація).
- **Combine (Об'єднання):** Результати обчислень з усіх окремих груп збираються назад в єдину підсумкову структуру даних (Series або DataFrame) із відповідними індексами груп.

**2. Чи розумієте принципову відмінність між pivot() і pivot_table() — чому один завершується помилкою на дублікатах комбінацій індекс/стовпець, а другий — ні?**
Метод `pivot()` призначений виключно для **зміни форми (структури) даних** за умови строго унікальних пар `(index, column)`. Вн не вміє узагальнювати дані. Метод `pivot_table()`, натомість, містить вбудований механізм агрегації (параметр `aggfunc`, який за замовчуванням дорівнює `'mean'`). При наявності декількох рядків для однієї комірки `pivot_table()` автоматично зводить їх до одного підсумкового значення (суми, середнього тощо), у той час як `pivot()` викликає виключення `ValueError`.

**3. Чи можете пояснити, що показує crosstab() і чим він відрізняється від еквівалентного результату через groupby(...).size()?**
`pd.crosstab()` обчислює **таблицю сопряженности (частотний розподіл)** двох або більше категоріальних змінних, показуючи кількість входжень (або відсотки) для кожної комбінації категорій у вигляді зручної двовимірної матриці (сітки). 
`groupby(['категорія1', 'категорія2']).size()` повертає точнісінько ті самі підраховані частоти, але у вигляді **одновимірного Series з мультиіндексом**. Для перетворення результату `groupby().size()` на вигляд `crosstab()` потрібно додатково викликати метод `.unstack()`.

**4. Чи можете пояснити, чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі, а не в таблиці з роками чи місяцями, розтягнутими в назви стовпців?**
У довгому (Tidy) форматі **кожен стовпець відповідає за одну окрему зміну (вектор)**, а **кожен рядок — за окреме спостереження**.
Якщо роки або місяці розтягнуті в назви стовпців (широкий формат), то назви стовпців містять у собі *значення даних*, а не *назви змінних*. У такому разі алгоритми `groupby()` або `pivot_table()` не можуть виконати групування за змінною «Рік», оскільки такої змінної як єдиного стовпця у таблиці просто не існує — дані роззосереджені по багатьох різних стовпчиках.